In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [ ]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [ ]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [ ]:
# model_type = 'llama'
# MODEL_VERSION='3.1'
# MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='3'
# MODEL_SIZE='1B'
# MODEL_SIZE='12B'

model_type = 'qwen'
MODEL_VERSION='3'
MODEL_SIZE='4B'
# MODEL_SIZE='8B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35]


Training

In [7]:
def read_single_translate(llm, fix_lang, other_langs, path, source_shift=True, hidden_layers=None):
    if hidden_layers is None:
        hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))

    Xt = {i: [] for i in hidden_layers}

    if source_shift:
        print("Source Constant")
        for other_lang in other_langs:
            tcontroller = NeuralController(
                llm,
                llm.tokenizer,
                rfm_iters=8,
                control_method="rfm",
                n_components=1
            )

            tcontroller.load_translate(concept=other_lang, model_name=llm.name, orig_lang=fix_lang, path=path)

            dirs = tcontroller.directions

            for k in Xt:
                Xt[k].append(dirs[k])
    else:
        print("Destination Constant")
        for other_lang in other_langs:
            tcontroller = NeuralController(
                llm,
                llm.tokenizer,
                rfm_iters=8,
                control_method="rfm",
                n_components=1
            )

            tcontroller.load_translate(concept=fix_lang, model_name=llm.name, orig_lang=other_lang, path=path)

            dirs = tcontroller.directions

            for k in Xt:
                Xt[k].append(dirs[k])
    
    
    X = {i: torch.cat(Xt[i]).to("cuda") for i in hidden_layers}
    
    return X

In [8]:
# all_langs = ['english', 'german', 'hindi', 'spanish', 'french', 'italian', 'portuguese', 'thai', 'chinese_simplified', 'chinese_traditional', 'japanese', 'korean', 'russian', 'arabic', 'vietnamese', 'indonesian', 'turkish', 'polish', 'dutch', 'ukrainian', 'czech', 'romanian', 'greek', 'hungarian', 'swedish', 'danish', 'finnish', 'norwegian', 'bulgarian', 'serbian', 'croatian', 'slovak', 'lithuanian', 'slovenian', 'latvian', 'estonian', 'catalan', 'hebrew', 'persian', 'tagalog', 'bengali', 'urdu', 'tamil', 'telugu', 'malayalam', 'kannada', 'marathi', 'gujarati', 'punjabi', 'malay', 'swahili', ]

In [8]:
# all_langs = ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']

all_langs = ['english', 'german', 'hindi', 'spanish', 'french', 'italian', 'portuguese', 'thai', 'chinese_simplified', 'chinese_traditional', 'japanese', 'korean', 'russian', 'arabic', 'vietnamese', 'indonesian', 'turkish', 'polish', 'dutch', 'ukrainian', 'czech',]



In [ ]:
orig_lang = 'english'
target_lang = 'italian'
# test_set = ['portuguese',]
test_set = ['japanese',]

# source_shift = True
source_shift = False

other_langs = [element for element in all_langs if element not in [orig_lang, target_lang] + test_set]

# orig_lang = 'french'
# target_lang = 'german'
# other_langs = ['english', 'italian', 'hindi', 'portuguese', 'spanish', 'thai']
# source_shift = True

# orig_lang = 'hindi'
# target_lang = 'thai'
# other_langs = ['english', 'italian', 'french', 'portuguese', 'spanish', 'german']
# source_shift = False

# path = '../all_gitignore/xRFM/language_multiex_llama/'
path = '../all_gitignore/xRFM/language_multiex_qwen/'

# X = read_single_translate(llm, orig_lang, other_langs, path, hidden_layers=hidden_layers)
# Y = read_single_translate(llm, target_lang, other_langs, path, hidden_layers=hidden_layers)

X = read_single_translate(llm, orig_lang, other_langs, path, source_shift=source_shift, hidden_layers=hidden_layers)
Y = read_single_translate(llm, target_lang, other_langs, path, source_shift=source_shift, hidden_layers=hidden_layers)

Destination Constant
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -

In [10]:
print(X[-2].shape)
print(Y[-2].shape)

torch.Size([18, 2560])
torch.Size([18, 2560])


In [11]:
big_X = torch.vstack(list(X.values())).cpu().numpy()
big_Y = torch.vstack(list(Y.values())).cpu().numpy()

In [12]:
print(big_X.shape)
print(big_Y.shape)

(630, 2560)
(630, 2560)


In [13]:
from sklearn.utils import shuffle

big_X_shuffled, big_Y_shuffled = shuffle(big_X, big_Y, random_state=SEED)

In [14]:
# model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=1000.0, print_error=True)
model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=np.arange(1, 4), print_error=True)

lrr_models = {i: model_lrr for i in hidden_layers}

Running with alpha: [  10.  100. 1000.]
Best lambda: 100.0
Training RMSE: 0.0015, Training R2: 0.9923
Done.


In [15]:
if source_shift:
    # with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/source_{orig_lang}TO{target_lang}-{test_set}.pkl', 'wb') as file:
    with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/qwen/source_{orig_lang}TO{target_lang}-{test_set}.pkl', 'wb') as file:
        pickle.dump(lrr_models, file)
else:
    # with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/dest_{orig_lang}TO{target_lang}-{test_set}.pkl', 'wb') as file:
    with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/qwen/dest_{orig_lang}TO{target_lang}-{test_set}.pkl', 'wb') as file:
        pickle.dump(lrr_models, file)

Testing

In [15]:
# coef = 0.2
# coef = 0.25
# coef = 0.3
# coef = 0.4
# coef = 0.45
coef = 0.5
# coef = 0.55
# coef = 0.65


max_tokens = 200

# ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']

orig_lang = 'english'
# orig_lang = 'hindi'

prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'

# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
l = 'portuguese'

l1_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/language_multi/')
orig_l = l1_controller.directions
out = test_concept_vector(l1_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens)

# l2_controller = load_controller_translate(llm, l, 'spanish', path=f'../all_gitignore/language_multi/')
# orig_l = l_controller.directions

# out = test_concept_vector(l2_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is weight of a football used in fifa?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The weight of a football used in FIFA is 410-450 grams (14.5-15.9 ounces) for the size 5 ball, which is the official size used in professional and international matches.<|eot_id|>

========================== + coef: 0.5, ques original: english, steered: portuguese Control (normal) ==========================
<|begin_of_t

Testing Source Shift

In [16]:
# Loading

orig_lang = 'english'
target_lang = 'italian'
# test_set = ['portuguese',]
test_set = ['japanese',]


# orig_lang = 'french'
# target_lang = 'german'

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/source_{orig_lang}TO{target_lang}-{test_set}.pkl', 'rb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/qwen/source_{orig_lang}TO{target_lang}-{test_set}.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [19]:
# coef = 0.45
# coef = 0.5
# coef = 0.55
# coef = 0.7 # llama

coef = 2.2
# coef = 2.3
# coef = 2.4
# coef = 2.5 # qwen


max_tokens = 200

prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'
# prompts = ["Qual è il peso di un pallone da calcio utilizzato in FIFA?"] # 'italian
# prompts = ["Quel est le poids d'un ballon de football utilisé à la Fifa ?"] # french
# prompts = ["Wie schwer ist ein Fußball, der in der FIFA verwendet wird?"] # german

# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
# l = 'portuguese'
l = 'japanese'

# l_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/language_multi/')
# l_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/xRFM/language_multiex_llama/')
l_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/xRFM/language_multiex_qwen/')
orig_l = l_controller.directions

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True)


new_prompts = ["Qual è il peso di un pallone da calcio utilizzato in FIFA?"] # 'italian

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, steered: {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, steered: {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, qwen=True)


trans_l = apply_auto(orig_l, lrr_models)
l_controller.directions = trans_l

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, source_M({orig_lang}to{target_lang})(steered): {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, source_M({orig_lang}to{target_lang})(steered): {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
 as of the latest FIFA regulations, the official weight of a football used in FIFA competitions (such as the World Cup) is:

**410 to 450 grams (14 to 16 ounces)**

This range applies to the official match ball used in FIFA-sanctioned games. The ball must also meet specific size and shape requirements:

- **Size**: 68–70 cm (27–28 inches) in circumference
- **Weight**: 410–450 grams (14–16 oz)
- **Surface**: Made of synthetic materials, with a smooth, consistent texture

These specifications ensure the ball per

In [23]:
coef = 2.6

trans_l = apply_auto(orig_l, lrr_models)
l_controller.directions = trans_l

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, source_M({orig_lang}to{target_lang})(steered): {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, source_M({orig_lang}to{target_lang})(steered): {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)


========================== + coef: 2.6, ques original: italian, source_M(englishtoitalian)(steered): japanese Control (normal) ==========================
Assistant: FIFA (国际足联) が定める **サッカーボール（サッカー用ボール）** の **重量** についてご説明します。

---

### ✅ FIFA が定める サッカーボール（ボール）の重量

FIFA が定める **サッカーボール**（正式な国際大会用ボール）の **重量** は以下の通りです：

- **重量（重量）**：**410g ～ 430g**（±10g）
- **直径**：22cm（±1cm）
- **素材**：合成皮革（合成生皮）または ポリウレタン（PU）製

---

### 🔍 なぜこの範囲か？

FIFA は、国際大会（例：ワールドカップ）で使用されるボールの **重量・サイズ・形状** を厳密に規定しています。

-


Testing Destination Shift

In [17]:
# Loading

orig_lang = 'english'
target_lang = 'italian'
# test_set = ['portuguese',]
test_set = ['japanese',]

# orig_lang = 'french'
# target_lang = 'german'

# orig_lang = 'hindi'
# target_lang = 'thai'

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/dest_{orig_lang}TO{target_lang}-{test_set}.pkl', 'rb') as file:
#     lrr_models = pickle.load(file)

In [19]:
# coef = 0.7
# coef = 0.75 # llama

coef = 2.5 # qwen


max_tokens = 200

# prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'
# prompts = ["Qual é o peso de uma bola de futebol usada na FIFA?"] # 'portuguese'
prompts = ["FIFAで使用されるサッカーボールの重さはどれくらいですか？"] # 'japanese'

# l = "english"
# l = "german"
# l = "hindi"
# l = "spanish"
# l = "thai"
# l = 'italian'
# l = 'portuguese'
l = 'japanese'

# l_controller = load_controller_translate(llm, orig_lang, l, path=f'../all_gitignore/language_multi/')
# l_controller = load_controller_translate(llm, orig_lang, l, path=f'../all_gitignore/xRFM/language_multiex_llama/')
l_controller = load_controller_translate(llm, orig_lang, l, path=f'../all_gitignore/xRFM/language_multiex_qwen/')
orig_l = l_controller.directions

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, steered: {orig_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, steered: {orig_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True)


# coef = 0.8

trans_l = apply_auto(orig_l, lrr_models)
l_controller.directions = trans_l

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, dest_M({orig_lang}to{target_lang})(steered): {target_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, dest_M({orig_lang}to{target_lang})(steered): {target_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)


Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
の回答は、FIFAが定めるサッカーボールの重さの基準について正確に説明しています。ただし、質問は「重さ」に焦点を当てており、回答では「重さ」だけでなく「直径」や「空気圧」なども含まれています。これは、FIFAの規則に従ってサッカーボールの特性を完全に理解するためには必要な情報ですが、質問の焦点である「重さ」に直接答える点で、少しズレがあります。

以下のように、質問に**直接的かつ正確に**答えるべきです：

---

**FIFAで使用されるサッカーボールの重さは、約410〜450グラム（g）です。**

これは、FIFAの公式規則（FIFA Regulations on the Equipment of the Game）に定められており、ボールの重さは**

========================== + coef: 2.5, ques original: japanese, steered: english Control (normal) ==========================
 Yes, the official soccer ball used in FIFA competiti

In [26]:
coef = 3.5

trans_l = apply_auto(orig_l, lrr_models)
l_controller.directions = trans_l

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, dest_M({orig_lang}to{target_lang})(steered): {target_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, dest_M({orig_lang}to{target_lang})(steered): {target_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)


========================== + coef: 3.5, ques original: portuguese, dest_M(englishtoitalian)(steered): italian Control (normal) ==========================

A FIFA (Fédération Internazionale di Calcio) **não define um peso preciso** per "bola di calcio" (bola de futebol), mas **definisce un range di peso** (intervalo de peso) per le palle usate in gara.

### Peso della palla di calcio (bola di calcio) in FIFA

Nel regolamento della **FIFA (Regolamento del Futsal)**, il peso della palla deve essere compreso tra:

> **410 e 450 grammi** (410 a 450 grammi)

Ovvero: **410 a 450 grammi** (410 a 450 grammi)

### In pratica: il peso della palla di calcio in campo

In pratica, il peso della palla di calcio in campo (bola


Sanity tests

In [ ]:
lang1 = "spanish"
lang2 = "english"

target_lang = "hindi"

l1_controller = load_controller_translate(llm, target_lang, lang1, path=f'../all_gitignore/language_multi/')
l2_controller = load_controller_translate(llm, target_lang, lang2, path=f'../all_gitignore/language_multi/')

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found


In [ ]:
compare_cosine(l1_controller.directions, l2_controller.directions)

layer: -1, cosine: 0.6345441341400146
layer: -2, cosine: 0.6440114378929138
layer: -3, cosine: 0.6595911979675293
layer: -4, cosine: 0.6843035221099854
layer: -5, cosine: 0.6993639469146729
layer: -6, cosine: 0.6909059882164001
layer: -7, cosine: 0.7007578611373901
layer: -8, cosine: 0.6582663059234619
layer: -9, cosine: 0.6346402168273926
layer: -10, cosine: 0.6194866299629211
layer: -11, cosine: 0.6064528226852417
layer: -12, cosine: 0.5957133769989014
layer: -13, cosine: 0.24182532727718353
layer: -14, cosine: 0.2508857250213623
layer: -15, cosine: 0.27932631969451904
layer: -16, cosine: 0.5888267755508423
layer: -17, cosine: 0.5975897312164307
layer: -18, cosine: 0.6290510296821594
layer: -19, cosine: 0.640220582485199
layer: -20, cosine: 0.6996731758117676
layer: -21, cosine: 0.7567381262779236
layer: -22, cosine: 0.7832663059234619
layer: -23, cosine: 0.788339376449585
layer: -24, cosine: 0.8005814552307129
layer: -25, cosine: 0.789667010307312
layer: -26, cosine: 0.8032115697860

0.660088257924203

In [ ]:
target_lang2 = "italian"

l3_controller = load_controller_translate(llm, target_lang2, lang1, path=f'../all_gitignore/language_multi/')
l4_controller = load_controller_translate(llm, target_lang2, lang2, path=f'../all_gitignore/language_multi/')

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found


In [18]:
compare_cosine(l4_controller.directions, l2_controller.directions)

layer: -1, cosine: 0.2026747167110443
layer: -2, cosine: 0.2745552062988281
layer: -3, cosine: 0.2866263687610626
layer: -4, cosine: 0.3091799020767212
layer: -5, cosine: 0.3644128441810608
layer: -6, cosine: 0.3524267077445984
layer: -7, cosine: 0.38730746507644653
layer: -8, cosine: 0.39211881160736084
layer: -9, cosine: 0.48528438806533813
layer: -10, cosine: 0.4989360570907593
layer: -11, cosine: 0.5511072278022766
layer: -12, cosine: 0.5706215500831604
layer: -13, cosine: 0.627822756767273
layer: -14, cosine: 0.6990546584129333
layer: -15, cosine: 0.6157951354980469
layer: -16, cosine: 0.6841753125190735
layer: -17, cosine: 0.7062395215034485
layer: -18, cosine: 0.6797906160354614
layer: -19, cosine: 0.7110946178436279
layer: -20, cosine: 0.7553110122680664
layer: -21, cosine: 0.7478247880935669
layer: -22, cosine: 0.7368202209472656
layer: -23, cosine: 0.7257872223854065
layer: -24, cosine: 0.7090833187103271
layer: -25, cosine: 0.7584625482559204
layer: -26, cosine: 0.7063909173

0.5509049892425537